# Moirai partial-unfreeze fine-tune (h = 6)

Upstream source: `Morai_improvement_setup.ipynb` in the thesis workbench.

**Input**: `DTW_DFS_DIR / 'dtw_20_dfs' / *.parquet`.
**Output**: `moirai_results_dir('dtw20')` - includes the fine-tuned checkpoint `moirai_ft_partial_h6.pt` (not distributed with the repo), finetune logs, and `moirai_engineered_metrics_dtw20_h6.csv`.

**Training budget**: 3-epoch 'make it move' schedule (MAE warm-start + NLL); partial parameter unfreeze on top of the pretrained Moirai module. Requires a GPU (tested on Colab T4).


## Canonical cell for the paper

The paper reports the **DTW-based Moirai** row (h = 6 partial-unfreeze fine-tune over DTW-neighbour covariates) from the cell titled *Canonical Moirai fine-tune (NLL on future)*. Other cells fall into two groups: (a) environment prep / package pinning needed on Colab to reach the canonical cell, and (b) baseline evaluation and canary sanity checks that verify but do not change the reported number.


In [ ]:
# --- Paper-repo bootstrap (replaces original Colab `drive.mount` cell) ---
import sys, os
from pathlib import Path

_REPO_SHARED = Path('..', '_shared').resolve()
if str(_REPO_SHARED) not in sys.path:
    sys.path.insert(0, str(_REPO_SHARED))

from paths import (  # noqa: E402
    ensure_env, DATA_ROOT, DTW_DFS_DIR, KEYWORDS_DIR_20, moirai_results_dir,
)
from api_keys import get_hf_token  # noqa: E402

ensure_env()
_hf_token = get_hf_token()
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token


In [ ]:
# Upgrade tooling
!pip install -U pip setuptools wheel

# Install PyTorch CUDA 12.1 wheels
!pip install --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1

# Install JAX GPU (CUDA 12 build)
!pip install "jax[cuda12]==0.4.26" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html


In [ ]:
# Create a copy without GPU framework pins to avoid conflicts
!grep -v -E '^(torch|torchvision|torchaudio|jax|jaxlib)=' requirements.txt > requirements_nogpu.txt

# (Optional but recommended) Faster installs & fewer rebuilds
!pip install -U pip setuptools wheel

# Install remaining dependencies
!pip install -r requirements_nogpu.txt


In [ ]:
import torch, jax, platform
print("Python:", platform.python_version())
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| torch.cuda:", torch.version.cuda)
print("JAX:", jax.__version__, "| devices:", jax.devices())


In [ ]:
!pip install -U "transformers>=4.46.0"


In [ ]:
# ==== Quick 3-epoch Moirai partial fine-tune & eval (Colab T4, stable loaders) ====
# - Self-contained: includes all helpers & definitions
# - Uses masked NLL on future horizon (positional API) + sMAPE@6 for validation
# - NUM_WORKERS=0 to avoid Colab multiprocessing stalls
# - Mixed precision (AMP) enabled; no global set_default_device('cuda')

import os, re, random, time, warnings
from dataclasses import dataclass
from typing import List, Tuple, Iterable
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

from uni2ts.model.moirai import MoiraiModule

# ---------------- Config ----------------
H             = 6
CTX           = 96
BATCH_SIZE    = 96           # try 128 if memory allows
LR_MAIN       = 2e-4         # a bit louder than 1e-4 for quick adaptation
WD            = 1e-4
EPOCHS        = 3            # <-- as requested
PATIENCE      = 2
K_LAST        = 2            # you can bump to 4 if needed
VAL_SPLIT     = 0.15
SEED          = 42
MODEL_ID      = "Salesforce/moirai-1.1-R-small"
USE_AMP       = True
NUM_WORKERS   = 0            # stable on Colab

# Paths (as you specified)
BASE    = DATA_ROOT
IN_DIR  = DTW_DFS_DIR / "dtw_20_dfs"    # individual keyword parquet/csv files
OUT_DIR = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PATH = OUT_DIR / "moirai_ft_partial_h6.pt"

print("Input dir:", IN_DIR)
print("Will save model to:", SAVE_PATH)

# ---------------- Repro & device ----------------
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    torch.set_default_device("cpu")    # keep CPU default; move batches explicitly
except Exception:
    pass
cudnn.benchmark = True
warnings.filterwarnings("ignore", module="lightning.fabric")  # optional noise

# ---------------- Data prep helpers ----------------
EXCLUDE_COLS = {"week","cpc_week","keyword","adcost_sum","adclicks_sum","year","week_num"}

def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-"); return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    s = p.suffix.lower()
    if s in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if s == ".csv": return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns: return None
    base = (
        df_pl.select(pl.col("week"),
                     pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
             .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
             .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    )
    exog_cols = [c for c in df_pl.columns if c not in EXCLUDE_COLS and c not in ("ds","y")]
    if exog_cols:
        exog = (df_pl.select(["week"] + exog_cols)
                     .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
                     .drop("week").sort("ds"))
        full = base.join(exog, on="ds", how="left")
        pdf = full.to_pandas()
        for c in exog_cols:
            if not pd.api.types.is_numeric_dtype(pdf[c]):
                pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
        pdf[exog_cols] = pdf[exog_cols].ffill().bfill()
        for c in exog_cols:
            if pdf[c].isna().any():
                med = np.nanmedian(pdf[c].astype(np.float32))
                pdf[c] = pdf[c].fillna(0.0 if np.isnan(med) else med)
            pdf[c] = pdf[c].astype(np.float32, copy=False)
    else:
        pdf = base.to_pandas()
    pdf["y"] = pdf["y"].astype(np.float32, copy=False)
    pdf = pdf.drop(columns=[c for c in pdf.columns if c in EXCLUDE_COLS], errors="ignore")
    keep = ["ds","y"] + [c for c in pdf.columns if c not in ("ds","y") and pd.api.types.is_numeric_dtype(pdf[c])]
    pdf = pdf[keep].sort_values("ds").reset_index(drop=True)
    return pdf if len(pdf) >= (CTX + H + 8) else None

def covars_all(pdf: pd.DataFrame) -> List[str]:
    return [c for c in pdf.columns if c not in ("ds","y") and pd.api.types.is_numeric_dtype(pdf[c])]

def load_all_series(in_dir: Path) -> List[pd.DataFrame]:
    files = sorted([p for p in in_dir.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
    out = []
    for p in tqdm(files, desc="Loading keyword series"):
        pdf = prep_keyword_pdf(p)
        if pdf is None: continue
        pdf["_series_name"] = p.stem
        out.append(pdf)
    return out

# ---------------- Windowing ----------------
@dataclass
class Window:
    y_past: np.ndarray
    y_true: np.ndarray
    x_past: np.ndarray
    x_fut:  np.ndarray

def make_windows_for_series(pdf: pd.DataFrame, covars_fixed: List[str]) -> List[Window]:
    y = pdf["y"].to_numpy(dtype=np.float32)
    T = len(pdf); D = len(covars_fixed)
    if D > 0:
        cols = [pdf[c].to_numpy(dtype=np.float32) if c in pdf.columns else np.zeros(T, np.float32)
                for c in covars_fixed]
        X = np.stack(cols, axis=0)
    else:
        X = None
    wins = []
    for t_end in range(CTX, T - H):
        y_past = y[t_end-CTX:t_end]; y_true = y[t_end:t_end+H]
        if D > 0:
            x_past = X[:, t_end-CTX:t_end]; x_fut = X[:, t_end:t_end+H]
        else:
            x_past = np.zeros((0, CTX), np.float32); x_fut = np.zeros((0, H), np.float32)
        wins.append(Window(y_past, y_true, x_past, x_fut))
    return wins

class RollingDataset(Dataset):
    def __init__(self, windows: List[Window]): self.w = windows
    def __len__(self): return len(self.w)
    def __getitem__(self, i):
        w = self.w[i]
        return (torch.from_numpy(w.y_past).float(),
                torch.from_numpy(w.x_past).float(),
                torch.from_numpy(w.x_fut).float(),
                torch.from_numpy(w.y_true).float())

# ---------------- Positional API builders ----------------
def build_positional_inputs(y_past: torch.Tensor, y_true: torch.Tensor):
    B, CTX_ = y_past.shape; H_ = y_true.shape[1]; total = CTX_ + H_
    target_full = torch.cat([y_past, y_true], dim=1).unsqueeze(-1)  # (B,total,1)
    observed_mask   = torch.zeros(B, total, 1, dtype=torch.bool, device=y_past.device)
    observed_mask[:, :CTX_, 0] = True
    sample_id       = torch.arange(B, device=y_past.device).unsqueeze(1).repeat(1, total)
    time_id         = torch.arange(total, device=y_past.device).unsqueeze(0).repeat(B, 1)
    variate_id      = torch.zeros(B, total, dtype=torch.long, device=y_past.device)
    prediction_mask = torch.zeros(B, total, dtype=torch.bool, device=y_past.device)
    prediction_mask[:, CTX_:] = True
    patch_size      = torch.ones(B, total, dtype=torch.long, device=y_past.device)
    return target_full, observed_mask, sample_id, time_id, variate_id, prediction_mask, patch_size

@torch.no_grad()
def predict_mean(module: MoiraiModule, y_past: torch.Tensor, horizon: int) -> torch.Tensor:
    B, CTX_ = y_past.shape; total = CTX_ + horizon
    target_full = torch.cat([y_past, torch.zeros(B, horizon, device=y_past.device)], dim=1).unsqueeze(-1)
    observed_mask = torch.zeros(B, total, 1, dtype=torch.bool, device=y_past.device)
    observed_mask[:, :CTX_, 0] = True
    sample_id  = torch.arange(B, device=y_past.device).unsqueeze(1).repeat(1, total)
    time_id    = torch.arange(total, device=y_past.device).unsqueeze(0).repeat(B, 1)
    variate_id = torch.zeros(B, total, dtype=torch.long, device=y_past.device)
    prediction_mask = torch.zeros(B, total, dtype=torch.bool, device=y_past.device); prediction_mask[:, CTX_:] = True
    patch_size = torch.ones(B, total, dtype=torch.long, device=y_past.device)
    dist = module(target_full, observed_mask, sample_id, time_id, variate_id, prediction_mask, patch_size)
    m = dist.mean
    if m.dim() == 3:  # (B,total,max_patch)
        m_future = m[:, CTX_:, :].mean(-1)   # point forecast via mean over patch dimension
    else:
        m_future = m[:, CTX_:]
    if m_future.size(-1) != horizon:
        m_future = m_future[..., -horizon:]
    return m_future  # (B,H)

def _expand_target_to_event(target_full: torch.Tensor, dist) -> torch.Tensor:
    m = dist.mean
    max_patch = m.shape[-1] if m.dim() >= 3 else 1
    if target_full.shape[-1] == 1 and max_patch > 1:
        target_full = target_full.expand(*target_full.shape[:-1], max_patch)  # (B,total,max_patch)
    return target_full

def smape_canonical(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    return float(100.0 * np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask]))

def eval_smape(module: MoiraiModule, loader) -> float:
    module.eval()
    ys, yh = [], []
    with torch.no_grad():
        for y_past, _xp, _xf, y_true in loader:
            y_past = y_past.to(device, non_blocking=True)
            yhat = predict_mean(module, y_past, H)  # (B,H)
            ys.append(y_true.numpy()); yh.append(yhat.cpu().numpy())
    ys = np.concatenate(ys, axis=0); yh = np.concatenate(yh, axis=0)
    return smape_canonical(ys, yh)

# ---------------- Load data ----------------
series_list = load_all_series(IN_DIR)
print(f"#series: {len(series_list)}"); assert len(series_list) > 0

covar_names_global = None
for pdf in series_list:
    cur = set(covars_all(pdf))
    covar_names_global = sorted(list(cur)) if covar_names_global is None else [c for c in covar_names_global if c in cur]
print(f"Global covariates (intersection) D = {len(covar_names_global)}")

all_windows: List[Window] = []
for pdf in series_list: all_windows.extend(make_windows_for_series(pdf, covar_names_global))
print(f"#windows: {len(all_windows)}"); assert len(all_windows) > 0

# Split
idx = np.arange(len(all_windows)); np.random.default_rng(SEED).shuffle(idx)
n_val = max(1, int(len(idx)*VAL_SPLIT))
val_idx, tr_idx = idx[:n_val], idx[n_val:]
train_ds = RollingDataset([all_windows[i] for i in tr_idx])
val_ds   = RollingDataset([all_windows[i] for i in val_idx])

g_cpu = torch.Generator(device='cpu'); g_cpu.manual_seed(SEED)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=NUM_WORKERS, pin_memory=(device=="cuda"),
    persistent_workers=False, generator=g_cpu
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    num_workers=NUM_WORKERS, pin_memory=(device=="cuda"),
    persistent_workers=False
)
print(f"[train] train={len(train_loader)} | val={len(val_loader)} | bs={BATCH_SIZE} | workers={NUM_WORKERS}")

# ---------------- Load model & (partial) unfreeze ----------------
def count_params(mod: torch.nn.Module) -> Tuple[int, int]:
    total = sum(p.numel() for p in mod.parameters())
    trainable = sum(p.numel() for p in mod.parameters() if p.requires_grad)
    return total, trainable

def freeze_all(mod: torch.nn.Module):
    for p in mod.parameters(): p.requires_grad = False

def detect_block_extrema(named_params: Iterable[Tuple[str, torch.nn.Parameter]]):
    enc_blocks = []
    pats = [r"encoder\.layers\.(\d+)\.", r"transformer\.encoder\.layers\.(\d+)\."]
    for n,_ in named_params:
        for pat in pats:
            m = re.search(pat, n)
            if m: enc_blocks.append(int(m.group(1)))
    enc_max = max(enc_blocks) if enc_blocks else None
    return enc_max, None

def unfreeze_last_k_blocks(mod: torch.nn.Module, k_last: int):
    enc_max, _ = detect_block_extrema(mod.named_parameters())
    def should_unfreeze(name: str) -> bool:
        if enc_max is None: return False
        for i in range(enc_max, enc_max-k_last, -1):
            if re.search(rf"(encoder|transformer\.encoder)\.layers\.{i}\.", name): return True
        return False
    for n,p in mod.named_parameters():
        if should_unfreeze(n): p.requires_grad = True
    # also unfreeze the obvious projections/heads
    for n,p in mod.named_parameters():
        if re.search(r"(in_proj|out_proj|proj|projection|final|lm_head|head)", n): p.requires_grad = True
        if re.search(r"(layernorm|layer_norm|norm)\.", n): p.requires_grad = True
    t,tr = count_params(mod)
    print(f"✅ Unfreeze summary: {tr:,}/{t:,} trainable ({tr/t*100:.2f}%)")

module = MoiraiModule.from_pretrained(MODEL_ID).to(device).train()
freeze_all(module)
unfreeze_last_k_blocks(module, K_LAST)

# ---------------- Optimizer / schedulers ----------------
trainable_params = [p for p in module.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR_MAIN, weight_decay=WD)

steps_per_epoch = max(1, len(train_loader))
warmup_steps = max(1, int(0.1 * EPOCHS * steps_per_epoch))
total_steps  = max(1, EPOCHS * steps_per_epoch)
scheduler = SequentialLR(
    optimizer,
    schedulers=[
        LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_steps),
        CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps))
    ],
    milestones=[warmup_steps]
)

scaler = GradScaler(enabled=(device=="cuda" and USE_AMP))

# ---------------- Train (3 epochs) ----------------
best_val = float("inf"); best_state=None; no_improve=0
print("🚀 Starting quick 3-epoch run...", flush=True)

for epoch in range(1, EPOCHS+1):
    print(f"[epoch {epoch}] init", flush=True)
    module.train()
    losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for step, (y_past, _x_past, _x_fut, y_true) in enumerate(pbar):
        if step == 0: print(f"[epoch {epoch}] first batch fetched", flush=True)

        y_past = y_past.to(device, non_blocking=True)
        y_true = y_true.to(device, non_blocking=True)
        target_full, observed_mask, sample_id, time_id, variate_id, prediction_mask, patch_size = \
            build_positional_inputs(y_past, y_true)

        with autocast(enabled=(device=="cuda" and USE_AMP)):
            dist = module(target_full, observed_mask, sample_id, time_id, variate_id, prediction_mask, patch_size)
            target_for_nll = _expand_target_to_event(target_full, dist)   # match dist event/patch dim
            logp = dist.log_prob(target_for_nll)                          # (B,total) or (B,total,1)
            if logp.dim() == 3: logp = logp.squeeze(-1)
            loss = -(logp[:, y_past.shape[1]:]).mean()                    # NLL on future only

        optimizer.zero_grad(set_to_none=True)
        if scaler.is_enabled():
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()

        scheduler.step()
        losses.append(loss.item())
        if step % 50 == 0:
            pbar.set_postfix(nll=np.mean(losses))

    val_smape = eval_smape(module, val_loader)
    train_nll = float(np.mean(losses)) if losses else float("nan")
    print(f"Epoch {epoch:02d} | train NLL={train_nll:.4f} | val sMAPE@6={val_smape:.2f}", flush=True)

    if val_smape + 1e-6 < best_val:
        best_val = val_smape
        best_state = {k: v.detach().cpu().clone() for k,v in module.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"⏹️ Early stopping at epoch {epoch} (best val sMAPE@6={best_val:.2f})")
            break

# ---------------- Save best ----------------
if best_state is not None:
    module.load_state_dict(best_state)
module.eval()
torch.save(module.state_dict(), SAVE_PATH)
print(f"✅ Saved partially fine-tuned Moirai to: {SAVE_PATH}")


In [ ]:
# ===== 3-epoch "make it move" finetune: MAE warm-start (1 ep) + NLL (2 ep) =====
import re, numpy as np, torch, torch.backends.cudnn as cudnn
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True
cudnn.benchmark = True

# --- Unfreeze more: last 6 encoder blocks + norms/heads
def _detect_enc_max(named_params):
    layers=[]
    for n,_ in named_params:
        m=re.search(r"(?:^|\.)(?:encoder|transformer\.encoder)\.layers\.(\d+)\.", n)
        if m: layers.append(int(m.group(1)))
    return max(layers) if layers else None

# freeze-all then selective unfreeze
for _,p in module.named_parameters(): p.requires_grad=False
enc_max = _detect_enc_max(module.named_parameters())
K_LAST = 6
def _should(name):
    ok=False
    if enc_max is not None:
        for i in range(enc_max, enc_max-K_LAST, -1):
            if re.search(rf"(encoder|transformer\.encoder)\.layers\.{i}\.", name):
                ok=True; break
    # also unfreeze input/output projections, heads, norms
    if re.search(r"(in_proj|out_proj|proj|projection|final|lm_head|head)", name): ok=True
    if re.search(r"(layernorm|layer_norm|norm)\.", name): ok=True
    return ok

for n,p in module.named_parameters():
    if _should(n): p.requires_grad=True

t = sum(p.numel() for _,p in module.named_parameters())
tr = sum(p.numel() for _,p in module.named_parameters() if p.requires_grad)
print(f"✅ Unfrozen params: {tr:,}/{t:,} ({100*tr/t:.2f}%)")

# --- Param groups: louder on heads/projections
decay, no_decay = [], []
head_like = []
for n,p in module.named_parameters():
    if not p.requires_grad: continue
    if re.search(r"(in_proj|out_proj|proj|projection|final|lm_head|head)", n):
        head_like.append(p); continue
    if p.ndim == 1 or n.endswith(".bias"):
        no_decay.append(p)
    else:
        decay.append(p)

LR_BASE = 5e-4       # louder base
WD = 1e-4
param_groups = [
    {"params": head_like, "lr": LR_BASE * 3.0, "weight_decay": 0.0},  # heads 3x, no WD
    {"params": decay,     "lr": LR_BASE,       "weight_decay": WD},
    {"params": no_decay,  "lr": LR_BASE,       "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(param_groups)
scaler = GradScaler(enabled=(device=="cuda" and USE_AMP))

# --- helper: expand target to patch/event dim for NLL
def _expand_target_to_event(target_full, dist):
    m = dist.mean
    max_patch = m.shape[-1] if m.dim() >= 3 else 1
    if target_full.shape[-1] == 1 and max_patch > 1:
        target_full = target_full.expand(*target_full.shape[:-1], max_patch)
    return target_full

# --- quick grad probe on first batch
module.train()
y_past, _xp, _xf, y_true = next(iter(train_loader))
y_past = y_past.to(device, non_blocking=True); y_true = y_true.to(device, non_blocking=True)
tf, om, sid, tid, vid, pm, ps = build_positional_inputs(y_past, y_true)
with autocast(enabled=(device=="cuda" and USE_AMP)):
    dist = module(tf, om, sid, tid, vid, pm, ps)
    t_nll = _expand_target_to_event(tf, dist)
    logp = dist.log_prob(t_nll);
    if logp.dim()==3: logp = logp.squeeze(-1)
    loss_probe = -(logp[:, y_past.shape[1]:]).mean()
optimizer.zero_grad(set_to_none=True)
if scaler.is_enabled():
    scaler.scale(loss_probe).backward()
else:
    loss_probe.backward()

printed=0; nonzero=0; total=0
for n,p in module.named_parameters():
    if p.requires_grad:
        total+=1
        if p.grad is not None and torch.isfinite(p.grad).all():
            g = p.grad.detach().abs().mean().item()
            if g>0: nonzero+=1
            if printed<10:
                print(f"grad|{n[:48]:48s} ~ {g:.3e}")
                printed+=1
print(f"nonzero-grad tensors: {nonzero}/{total}")
optimizer.zero_grad(set_to_none=True)

# --- EPOCH 1: MAE warm-start on point means (makes things move)
print("\n🧪 Epoch 1/3: MAE warm-start")
mae = torch.nn.L1Loss()
module.train(); losses=[]
for step,(y_past,_x,_f,y_true) in enumerate(tqdm(train_loader, leave=False, desc="MAE 1/1")):
    y_past=y_past.to(device, non_blocking=True); y_true=y_true.to(device, non_blocking=True)
    tf, om, sid, tid, vid, pm, ps = build_positional_inputs(y_past, torch.zeros_like(y_true))  # future not needed
    with autocast(enabled=(device=="cuda" and USE_AMP)):
        dist = module(tf, om, sid, tid, vid, pm, ps)
        m = dist.mean
        yhat = (m[:, y_past.shape[1]:, :].mean(-1) if m.dim()==3 else m[:, y_past.shape[1]:])  # (B,H)
        loss = mae(yhat, y_true)
    optimizer.zero_grad(set_to_none=True)
    if scaler.is_enabled():
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_([p for g in param_groups for p in g["params"]], 1.0)
        scaler.step(optimizer); scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for g in param_groups for p in g["params"]], 1.0)
        optimizer.step()
    losses.append(float(loss.detach().cpu()))
print(f"MAE warm-start | train MAE={np.mean(losses):.4f}")

# --- quick eval after warm-start
@torch.no_grad()
def _eval_smape(loader):
    module.eval()
    ys, yh = [], []
    for y_past, _xp, _xf, y_true in loader:
        y_past = y_past.to(device, non_blocking=True)
        # predict_mean must output (B,H)
        B, CTX_ = y_past.shape; total = CTX_ + H
        tf = torch.cat([y_past, torch.zeros(B, H, device=y_past.device)], dim=1).unsqueeze(-1)
        om = torch.zeros(B, total, 1, dtype=torch.bool, device=y_past.device); om[:, :CTX_, 0] = True
        sid = torch.arange(B, device=y_past.device).unsqueeze(1).repeat(1, total)
        tid = torch.arange(total, device=y_past.device).unsqueeze(0).repeat(B, 1)
        vid = torch.zeros(B, total, dtype=torch.long, device=y_past.device)
        pm  = torch.zeros(B, total, dtype=torch.bool, device=y_past.device); pm[:, CTX_:] = True
        ps  = torch.ones(B, total, dtype=torch.long, device=y_past.device)
        dist = module(tf, om, sid, tid, vid, pm, ps)
        m = dist.mean
        yhat = (m[:, CTX_:, :].mean(-1) if m.dim()==3 else m[:, CTX_:])
        ys.append(y_true.numpy()); yh.append(yhat.cpu().numpy())
    ys = np.concatenate(ys, axis=0); yh = np.concatenate(yh, axis=0)
    denom = (np.abs(ys)+np.abs(yh))/2.0
    mask = denom!=0
    return float(100.0*np.mean(np.abs(yh[mask]-ys[mask])/denom[mask]))

sm_warm = _eval_smape(val_loader)
print(f"Post-warm-start val sMAPE@6 = {sm_warm:.2f}")

# --- EPOCHS 2-3: NLL (masked future)
print("\n🎯 Epochs 2–3: NLL training")
def train_nll_one_epoch(ep):
    module.train(); losses=[]
    pbar=tqdm(train_loader, leave=False, desc=f"NLL {ep}/2")
    for y_past,_x,_f,y_true in pbar:
        y_past=y_past.to(device, non_blocking=True); y_true=y_true.to(device, non_blocking=True)
        tf, om, sid, tid, vid, pm, ps = build_positional_inputs(y_past, y_true)
        with autocast(enabled=(device=="cuda" and USE_AMP)):
            dist = module(tf, om, sid, tid, vid, pm, ps)
            t_nll = _expand_target_to_event(tf, dist)
            logp = dist.log_prob(t_nll)
            if logp.dim()==3: logp=logp.squeeze(-1)
            loss = -(logp[:, y_past.shape[1]:]).mean()
        optimizer.zero_grad(set_to_none=True)
        if scaler.is_enabled():
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_([p for g in param_groups for p in g["params"]], 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for g in param_groups for p in g["params"]], 1.0)
            optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

nll2 = train_nll_one_epoch(1); sm2 = _eval_smape(val_loader)
print(f"NLL ep1 | train NLL={nll2:.4f} | val sMAPE@6={sm2:.2f}")
nll3 = train_nll_one_epoch(2); sm3 = _eval_smape(val_loader)
print(f"NLL ep2 | train NLL={nll3:.4f} | val sMAPE@6={sm3:.2f}")

# --- Save checkpoint
module.eval()
torch.save(module.state_dict(), SAVE_PATH)
print(f"\n✅ Saved to: {SAVE_PATH}")


In [ ]:
# Remove TB so it can't be imported
!pip uninstall -y tensorboard tensorboard-data-server tb-nightly

# Satisfy Colab’s preinstalls (TF/gcsfs/grpcio-status) and our stack
!pip install -U "protobuf>=5.29,<6" "fsspec==2025.3.2" \
  "lightning>=2.4,<3" "transformers>=4.46,<5" "uni2ts>=0.2.3" "torchmetrics<1.5"


#i think here starts the setup with finetuner cause morais internal version is scuffed

In [ ]:
!pip uninstall -y tensorboard tensorboard-data-server tb-nightly || true
!pip install -U --no-deps protobuf==5.29.0 fsspec==2025.3.2
!pip install -U --no-deps lightning==2.4.0 transformers==4.46.2 torchmetrics==1.4.0.post0 uni2ts==0.2.3


In [ ]:
!pip install -U --no-deps \
  lightning-utilities==0.11.7 \
  PyYAML==6.0.2 \
  packaging==24.2 \
  tqdm==4.66.5 \
  numpy


In [ ]:
!pip install -U --no-deps tokenizers==0.20.3
!pip install -U --no-deps jaxtyping==0.2.33
!pip install -U --no-deps gluonts==0.14.3
!pip install -U --no-deps hydra-core==1.3.2


In [ ]:
import torch, lightning as L
from uni2ts.model.moirai import MoiraiModule
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU avail:", torch.cuda.is_available())
print("Lightning:", L.__version__)
print("MoiraiModule OK")


### ↓ Canonical cell for the paper (see banner at the top of the notebook)

Below is the cell whose output is reported in the paper. All earlier cells in this notebook run, pin packages, or iterate the methodology that leads up to this cell.


In [ ]:
# ==== Canonical Moirai fine-tune (NLL on future) — Colab GPU ====
import re, numpy as np, torch, polars as pl, pandas as pd
from datetime import date
from pathlib import Path
from dataclasses import dataclass
from typing import List
from torch.utils.data import Dataset, DataLoader
import lightning as L
from lightning.pytorch.loggers import CSVLogger
from torch.amp import autocast, GradScaler
from uni2ts.model.moirai import MoiraiModule

# ----- config / paths -----
H, CTX = 6, 96
BATCH_SIZE, NUM_WORKERS = 96, 2
MODEL_ID = "Salesforce/moirai-1.1-R-small"
LAST_K, LR_BASE, WD, MAX_EPOCHS = 4, 2e-4, 1e-4, 3
BASE = DATA_ROOT
IN_DIR = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20"); OUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PATH = OUT_DIR / "moirai_finetune_h6.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Input dir:", IN_DIR)
print("Save path:", SAVE_PATH)

# ----- data helpers (your schema) -----
EXCLUDE_COLS = {"week","cpc_week","keyword","adcost_sum","adclicks_sum","year","week_num"}
def iso_week_to_date(wwyyyy: str):
    w,y = wwyyyy.split("-"); return date.fromisocalendar(int(y), int(w), 1)
def _read_pl_any(p: Path) -> pl.DataFrame:
    s=p.suffix.lower()
    if s in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if s==".csv": return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")
def prep_keyword_pdf(p: Path):
    df = _read_pl_any(p)
    if "week" not in df.columns or "cpc_week" not in df.columns: return None
    base = (df.select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
              .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
              .drop("week").sort("ds").filter(pl.col("y").is_not_null()))
    exog_cols = [c for c in df.columns if c not in EXCLUDE_COLS and c not in ("ds","y")]
    if exog_cols:
        exog = (df.select(["week"]+exog_cols)
                  .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
                  .drop("week").sort("ds"))
        pdf = base.join(exog, on="ds", how="left").to_pandas()
        for c in exog_cols:
            if not pd.api.types.is_numeric_dtype(pdf[c]): pdf[c]=pd.to_numeric(pdf[c], errors="coerce")
        pdf[exog_cols]=pdf[exog_cols].ffill().bfill().astype(np.float32, copy=False)
    else:
        pdf = base.to_pandas()
    pdf["y"]=pdf["y"].astype(np.float32, copy=False)
    pdf = pdf.drop(columns=[c for c in pdf.columns if c in EXCLUDE_COLS], errors="ignore")
    keep = ["ds","y"]+[c for c in pdf.columns if c not in ("ds","y") and pd.api.types.is_numeric_dtype(pdf[c])]
    pdf = pdf[keep].sort_values("ds").reset_index(drop=True)
    return pdf if len(pdf) >= (CTX+H+8) else None

@dataclass
class Window: y_past: np.ndarray; y_true: np.ndarray
def make_windows_for_series(pdf: pd.DataFrame) -> List[Window]:
    y=pdf["y"].to_numpy(np.float32); T=len(pdf); wins=[]
    for t_end in range(CTX, T-H):
        wins.append(Window(y[t_end-CTX:t_end], y[t_end:t_end+H]))
    return wins

class RollingDataset(Dataset):
    def __init__(self, wins): self.w=wins
    def __len__(self): return len(self.w)
    def __getitem__(self,i):
        w=self.w[i]
        return torch.from_numpy(w.y_past).float(), torch.from_numpy(w.y_true).float()

# ----- build loaders -----
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
series = [pdf for p in files if (pdf:=prep_keyword_pdf(p)) is not None]
wins=[]; [wins.extend(make_windows_for_series(pdf)) for pdf in series]
idx = np.arange(len(wins)); np.random.default_rng(42).shuffle(idx)
n_val = max(1,int(0.15*len(idx))); val_idx, tr_idx = idx[:n_val], idx[n_val:]
train_ds = RollingDataset([wins[i] for i in tr_idx]); val_ds = RollingDataset([wins[i] for i in val_idx])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda"))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda"))
print(f"[loaders] train={len(train_loader)} | val={len(val_loader)} | bs={BATCH_SIZE} | workers={NUM_WORKERS}")

# ----- load base + partial unfreeze (last-K + heads/norms) -----
module = MoiraiModule.from_pretrained(MODEL_ID).to(device).train()
def _detect_enc_max(named_params):
    layers=[]
    for n,_ in named_params:
        m=re.search(r"(?:^|\.)(?:encoder|transformer\.encoder)\.layers\.(\d+)\.", n)
        if m: layers.append(int(m.group(1)))
    return max(layers) if layers else None
for _,p in module.named_parameters(): p.requires_grad=False
enc_max=_detect_enc_max(module.named_parameters())
def _should(name):
    if enc_max is not None:
        for i in range(enc_max, enc_max-LAST_K, -1):
            if re.search(rf"(encoder|transformer\.encoder)\.layers\.{i}\.", name): return True
    if re.search(r"(in_proj|out_proj|proj|projection|final|lm_head|head)", name): return True
    if re.search(r"(layernorm|layer_norm|norm)\.", name): return True
    return False
for n,p in module.named_parameters():
    if _should(n): p.requires_grad=True
trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
total = sum(p.numel() for p in module.parameters())
print(f"✅ Trainable params: {trainable:,}/{total:,}")

# ----- Lightning wrapper with canonical NLL -----
class CompatMoiraiFinetune(L.LightningModule):
    def __init__(self, model, lr=LR_BASE, weight_decay=WD, horizon=H):
        super().__init__(); self.model=model; self.lr=lr; self.wd=weight_decay; self.H=horizon
        self.scaler = GradScaler('cuda', enabled=(self.device.type=='cuda'))
    def configure_optimizers(self):
        decay,no_decay,head_like=[],[],[]
        for n,p in self.model.named_parameters():
            if not p.requires_grad: continue
            if re.search(r"(in_proj|out_proj|proj|projection|final|lm_head|head)", n): head_like.append(p); continue
            if p.ndim==1 or n.endswith(".bias") or "norm" in n.lower(): no_decay.append(p)
            else: decay.append(p)
        return torch.optim.AdamW([
            {"params": head_like, "lr": self.lr*3.0, "weight_decay": 0.0},
            {"params": decay,     "lr": self.lr,     "weight_decay": self.wd},
            {"params": no_decay,  "lr": self.lr,     "weight_decay": 0.0},
        ])
    @staticmethod
    def _build_inputs_for_prediction(y_past, H):
        B,CTX=y_past.shape; total=CTX+H
        target_in=torch.cat([y_past, torch.zeros(B,H,device=y_past.device)],1).unsqueeze(-1)
        observed_mask=torch.zeros(B,total,1,dtype=torch.bool,device=y_past.device); observed_mask[:,:CTX,0]=True
        sample_id=torch.arange(B,device=y_past.device).unsqueeze(1).repeat(1,total)
        time_id=torch.arange(total,device=y_past.device).unsqueeze(0).repeat(B,1)
        variate_id=torch.zeros(B,total,dtype=torch.long,device=y_past.device)
        prediction_mask=torch.zeros(B,total,dtype=torch.bool,device=y_past.device); prediction_mask[:,CTX:]=True
        patch_size=torch.ones(B,total,dtype=torch.long,device=y_past.device)
        return target_in, observed_mask, sample_id, time_id, variate_id, prediction_mask, patch_size
    @staticmethod
    def _expand_value_to_event(value_full, dist):
        m=dist.mean; max_patch=m.shape[-1] if m.dim()>=3 else 1
        return value_full.expand(*value_full.shape[:-1], max_patch) if (value_full.shape[-1]==1 and max_patch>1) else value_full
    def training_step(self, batch, batch_idx):
        y_past, y_true = batch
        y_past=y_past.to(self.device, non_blocking=True); y_true=y_true.to(self.device, non_blocking=True)
        t_in,om,sid,tid,vid,pm,ps=self._build_inputs_for_prediction(y_past,self.H)
        with autocast('cuda', enabled=(self.device.type=='cuda')):
            dist=self.model(t_in,om,sid,tid,vid,pm,ps)
            value_full=torch.cat([y_past,y_true],1).unsqueeze(-1); value_full=self._expand_value_to_event(value_full,dist)
            logp=dist.log_prob(value_full); logp=logp.squeeze(-1) if logp.dim()==3 else logp
            loss=-(logp[:, y_past.shape[1]:]).mean()
        self.log("train_nll", loss, prog_bar=True, on_step=False, on_epoch=True, batch_size=y_past.size(0))
        return loss
    def validation_step(self, batch, batch_idx):
        y_past, y_true = batch
        y_past=y_past.to(self.device, non_blocking=True); y_true=y_true.to(self.device, non_blocking=True)
        t_in,om,sid,tid,vid,pm,ps=self._build_inputs_for_prediction(y_past,self.H)
        with torch.no_grad(), autocast('cuda', enabled=(self.device.type=='cuda')):
            dist=self.model(t_in,om,sid,tid,vid,pm,ps)
            m=dist.mean
            yhat=(m[:, y_past.shape[1]:, :].mean(-1) if m.dim()==3 else m[:, y_past.shape[1]:])[:, :self.H]
        ys=y_true.detach().cpu().numpy(); yh=yhat.detach().cpu().numpy()
        denom=(np.abs(ys)+np.abs(yh))/2.0; mask=denom!=0
        smape=float(100.0*np.mean(np.abs(yh[mask]-ys[mask])/denom[mask]))
        self.log("val_sMAPE@6", smape, prog_bar=True, on_step=False, on_epoch=True, batch_size=y_past.size(0))

ft = CompatMoiraiFinetune(module, lr=LR_BASE, weight_decay=WD, horizon=H)
logger = CSVLogger(save_dir="lightning_logs", name="moirai_finetune")
trainer = L.Trainer(max_epochs=MAX_EPOCHS,
                    accelerator=("gpu" if device.type=="cuda" else "cpu"),
                    devices=1,
                    precision=("16-mixed" if device.type=="cuda" else "32-true"),
                    logger=logger,
                    gradient_clip_val=1.0,
                    log_every_n_steps=25,
                    enable_progress_bar=True)
trainer.fit(ft, train_dataloaders=train_loader, val_dataloaders=val_loader)

# save
module.eval().to("cpu")
torch.save(module.state_dict(), SAVE_PATH)
print(f"✅ Saved to: {SAVE_PATH}")


In [ ]:
# Baseline evaluation (no finetune): sMAPE on your val_ds
import numpy as np, torch
from torch.utils.data import DataLoader
from uni2ts.model.moirai import MoiraiModule

H = 6
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# assumes val_ds exists; if not, re-create loaders as in your training cell
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

def positional_predict_mean(model, y_past, H):
    B, CTX = y_past.shape; total = CTX + H
    target_in = torch.cat([y_past, torch.zeros(B,H,device=y_past.device)],1).unsqueeze(-1)
    observed_mask = torch.zeros(B,total,1,dtype=torch.bool,device=y_past.device); observed_mask[:,:CTX,0] = True
    sample_id  = torch.arange(B, device=y_past.device).unsqueeze(1).repeat(1,total)
    time_id    = torch.arange(total, device=y_past.device).unsqueeze(0).repeat(B,1)
    variate_id = torch.zeros(B,total,dtype=torch.long,device=y_past.device)
    prediction_mask = torch.zeros(B,total,dtype=torch.bool,device=y_past.device); prediction_mask[:,CTX:] = True
    patch_size = torch.ones(B,total,dtype=torch.long,device=y_past.device)
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
        dist = model(target_in, observed_mask, sample_id, time_id, variate_id, prediction_mask, patch_size)
        m = dist.mean
        yhat = (m[:, CTX:, :].mean(-1) if m.dim()==3 else m[:, CTX:])
    return yhat

# load fresh pretrained (frozen) model
frozen = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

ys_all, yh_all = [], []
for y_past, y_true in val_loader:
    y_past = y_past.to(device)
    yh = positional_predict_mean(frozen, y_past, H)[:, :H]
    ys_all.append(y_true.numpy()); yh_all.append(yh.cpu().numpy())

ys = np.concatenate(ys_all, 0); yh = np.concatenate(yh_all, 0)
den = (np.abs(ys) + np.abs(yh))/2.0; m = den != 0
smape_base = float(100.0*np.mean(np.abs(yh[m]-ys[m])/den[m]))
print(f"[BASELINE] Frozen Moirai val sMAPE@{H} = {smape_base:.2f}")


In [ ]:
# --- Fix eval device mismatch + rerun probes quickly ---

import torch, numpy as np, re, copy, lightning as L
from torch.utils.data import DataLoader
from lightning.pytorch.loggers import CSVLogger
from uni2ts.model.moirai import MoiraiModule

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
H = 6

# Reuse your existing train_ds / val_ds
train_loader_probe = DataLoader(train_ds, batch_size=96, shuffle=True,  drop_last=True,  num_workers=0)
val_loader_probe   = DataLoader(val_ds,   batch_size=128, shuffle=False, drop_last=False, num_workers=0)

def positional_predict_mean(model, y_past, H):
    B, CTX = y_past.shape; total = CTX + H
    target_in = torch.cat([y_past, torch.zeros(B, H, device=y_past.device)], 1).unsqueeze(-1)
    observed_mask = torch.zeros(B, total, 1, dtype=torch.bool, device=y_past.device); observed_mask[:, :CTX, 0] = True
    sample_id  = torch.arange(B, device=y_past.device).unsqueeze(1).repeat(1, total)
    time_id    = torch.arange(total, device=y_past.device).unsqueeze(0).repeat(B, 1)
    variate_id = torch.zeros(B, total, dtype=torch.long, device=y_past.device)
    prediction_mask = torch.zeros(B, total, dtype=torch.bool, device=y_past.device); prediction_mask[:, CTX:] = True
    patch_size = torch.ones(B, total, dtype=torch.long, device=y_past.device)
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=(y_past.device.type=='cuda')):
        dist = model(target_in, observed_mask, sample_id, time_id, variate_id, prediction_mask, patch_size)
        m = dist.mean
        yhat = (m[:, CTX:, :].mean(-1) if m.dim()==3 else m[:, CTX:])[:, :H]
    return yhat

def eval_smape(model_eval):
    # 🔧 ensure model on same device as batches
    model_eval = model_eval.to(device).eval()
    ys_all, yh_all = [], []
    for y_past, y_true in val_loader_probe:
        y_past = y_past.to(device, non_blocking=True)
        yh = positional_predict_mean(model_eval, y_past, H)
        ys_all.append(y_true.numpy()); yh_all.append(yh.cpu().numpy())
    ys = np.concatenate(ys_all, 0); yh = np.concatenate(yh_all, 0)
    den = (np.abs(ys)+np.abs(yh))/2.0; m = den != 0
    return float(100.0*np.mean(np.abs(yh[m]-ys[m]) / den[m]))

class ProbeFinetune(L.LightningModule):
    def __init__(self, model, lr=2e-4, wd=1e-4, horizon=H):
        super().__init__(); self.model=model; self.lr=lr; self.wd=wd; self.H=horizon
    def configure_optimizers(self):
        decay,no_decay,head_like=[],[],[]
        for n,p in self.model.named_parameters():
            if not p.requires_grad: continue
            if re.search(r"(in_proj|out_proj|proj|projection|final|lm_head|head)", n): head_like.append(p); continue
            if p.ndim==1 or n.endswith(".bias") or "norm" in n.lower(): no_decay.append(p)
            else: decay.append(p)
        return torch.optim.AdamW([
            {"params": head_like, "lr": self.lr*3.0, "weight_decay": 0.0},
            {"params": decay,     "lr": self.lr,     "weight_decay": self.wd},
            {"params": no_decay,  "lr": self.lr,     "weight_decay": 0.0},
        ])
    def training_step(self, batch, _):
        y_past, y_true = batch
        y_past=y_past.to(self.device, non_blocking=True); y_true=y_true.to(self.device, non_blocking=True)
        # build positional inputs (prediction mask on future)
        B, CTX = y_past.shape; total = CTX + self.H
        target_in = torch.cat([y_past, torch.zeros(B,self.H,device=y_past.device)], 1).unsqueeze(-1)
        observed_mask = torch.zeros(B,total,1,dtype=torch.bool,device=y_past.device); observed_mask[:,:CTX,0]=True
        sample_id  = torch.arange(B,device=y_past.device).unsqueeze(1).repeat(1,total)
        time_id    = torch.arange(total,device=y_past.device).unsqueeze(0).repeat(B,1)
        variate_id = torch.zeros(B,total,dtype=torch.long,device=y_past.device)
        prediction_mask = torch.zeros(B,total,dtype=torch.bool,device=y_past.device); prediction_mask[:,CTX:] = True
        patch_size = torch.ones(B,total,dtype=torch.long,device=y_past.device)
        with torch.amp.autocast('cuda', enabled=(self.device.type=='cuda')):
            dist = self.model(target_in, observed_mask, sample_id, time_id, variate_id, prediction_mask, patch_size)
            value_full = torch.cat([y_past, y_true], 1).unsqueeze(-1)
            # expand to event dim if needed
            m = dist.mean; max_patch = m.shape[-1] if m.dim()>=3 else 1
            if value_full.shape[-1]==1 and max_patch>1:
                value_full = value_full.expand(*value_full.shape[:-1], max_patch)
            logp = dist.log_prob(value_full)
            if logp.dim()==3: logp = logp.squeeze(-1)
            loss = -(logp[:, CTX:]).mean()
        self.log("train_nll", loss, prog_bar=False, on_step=False, on_epoch=True, batch_size=y_past.size(0))
        return loss

def unfreeze_last_k(mod, k_last=None):
    for _,p in mod.named_parameters(): p.requires_grad=False
    enc_max=None
    for n,_ in mod.named_parameters():
        m=re.search(r"(?:^|\.)(?:encoder|transformer\.encoder)\.layers\.(\d+)\.", n)
        if m:
            i=int(m.group(1))
            enc_max = i if enc_max is None else max(enc_max, i)
    def should(n):
        if re.search(r"(in_proj|out_proj|proj|projection|final|lm_head|head)", n): return True
        if re.search(r"(layernorm|layer_norm|norm)\.", n): return True
        if k_last is None: return True  # unfreeze all encoder blocks
        if enc_max is None: return False
        for i in range(enc_max, enc_max-k_last, -1):
            if re.search(rf"(encoder|transformer\.encoder)\.layers\.{i}\.", n): return True
        return False
    for n,p in mod.named_parameters():
        if should(n): p.requires_grad=True

# cache fresh base weights
_base = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device)
base_state = {k: v.detach().cpu() for k,v in _base.state_dict().items()}

def run_probe(k_last, lr=2e-4, max_train_batches=50):
    model = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).train()
    model.load_state_dict(base_state, strict=False)
    unfreeze_last_k(model, k_last=k_last)
    ft = ProbeFinetune(model, lr=lr, wd=1e-4, horizon=H)
    trainer = L.Trainer(
        max_epochs=1, accelerator=("gpu" if device.type=="cuda" else "cpu"), devices=1,
        precision=("16-mixed" if device.type=="cuda" else "32-true"),
        logger=CSVLogger("lightning_logs", name=f"probe_k{k_last if k_last is not None else 'ALL'}_lr{lr}"),
        gradient_clip_val=1.0, limit_train_batches=max_train_batches, limit_val_batches=0,  # train-only
        enable_progress_bar=True
    )
    trainer.fit(ft, train_dataloaders=train_loader_probe)  # no val loop during fit
    # 🔧 ensure eval on the right device
    return eval_smape(model)

# (Re)run quick probes
print("Re-running two quick probes with device-safe eval…")
probe1 = run_probe(k_last=2,  lr=2e-4, max_train_batches=50)
probe2 = run_probe(k_last=4,  lr=2e-4, max_train_batches=50)
print(f"[Probe] LAST_K=2  → val sMAPE@6 = {probe1:.2f}")
print(f"[Probe] LAST_K=4  → val sMAPE@6 = {probe2:.2f}")


In [ ]:
# ===== Canary fine-tune verifier (tiny slices, strong settings) =====
import re, numpy as np, torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from uni2ts.model.moirai import MoiraiModule
from torch.amp import autocast

# ---- knobs ----
H, CTX       = 6, 96
TRAIN_N      = 2000
VAL_N        = 600
STEPS        = 250          # short, but strong
BS           = 128
LR_BACK      = 3e-4         # backbone LR
LR_HEAD      = 1e-3         # heads LR
WEIGHT_DECAY = 0.0
USE_AMP      = False        # keep OFF for this verifier
SEED         = 777

np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- build fixed tiny slices from your existing train_ds ----
all_idx = np.arange(len(train_ds))
np.random.shuffle(all_idx)
tiny_train_idx = all_idx[:TRAIN_N]
tiny_val_idx   = all_idx[TRAIN_N:TRAIN_N+VAL_N]
tiny_train = Subset(train_ds, tiny_train_idx)
tiny_val   = Subset(train_ds, tiny_val_idx)

train_loader = DataLoader(tiny_train, batch_size=BS,   shuffle=True,  drop_last=True,  num_workers=0, pin_memory=(device.type=="cuda"))
val_loader   = DataLoader(tiny_val,   batch_size=256,  shuffle=False, drop_last=False, num_workers=0, pin_memory=(device.type=="cuda"))

# ---- fresh model, unfreeze ALL encoder blocks + heads/norms ----
model = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).train()
for _, p in model.named_parameters(): p.requires_grad = False

def should_unfreeze(n: str):
    if re.search(r"(encoder|transformer\.encoder)\.layers\.", n): return True
    if re.search(r"(in_proj|out_proj|proj|projection|final|lm_head|head)", n): return True
    if "norm" in n.lower(): return True
    return False

for n, p in model.named_parameters():
    if should_unfreeze(n):
        p.requires_grad = True

# Print how much is trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ trainable params: {trainable:,}/{total:,}")

# ---- split params into groups ----
decay, no_decay, heads = [], [], []
for n,p in model.named_parameters():
    if not p.requires_grad: continue
    if re.search(r"(in_proj|out_proj|proj|projection|final|lm_head|head)", n):
        heads.append(p)
    elif p.ndim == 1 or n.endswith(".bias") or "norm" in n.lower():
        no_decay.append(p)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW([
    {"params": heads,    "lr": LR_HEAD, "weight_decay": 0.0},
    {"params": decay,    "lr": LR_BACK, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay, "lr": LR_BACK, "weight_decay": 0.0},
])
loss_mae = nn.L1Loss()
scaler   = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda" and USE_AMP))

def build_inputs(y_past, y_true, horizon):
    """Teacher forcing for future: feed true future into target tensor."""
    B, CTX_ = y_past.shape; total = CTX_ + horizon
    tgt = torch.cat([y_past, y_true], 1).unsqueeze(-1)   # <— TRUE future, not zeros
    obs = torch.zeros(B,total,1,dtype=torch.bool,device=y_past.device); obs[:,:CTX_,0]=True
    sid = torch.arange(B,device=y_past.device).unsqueeze(1).repeat(1,total)
    tid = torch.arange(total,device=y_past.device).unsqueeze(0).repeat(B,1)
    vid = torch.zeros(B,total,dtype=torch.long,device=y_past.device)
    pmsk = torch.zeros(B,total,dtype=torch.bool,device=y_past.device); pmsk[:,CTX_:]=True
    psz  = torch.ones(B,total,dtype=torch.long,device=y_past.device)
    return tgt, obs, sid, tid, vid, pmsk, psz

@torch.no_grad()
def eval_smape_on(loader):
    model.eval()
    ys_all, yh_all = [], []
    for y_past, y_true in loader:
        y_past = y_past.to(device, non_blocking=True)
        # inference uses zeros in future (proper forecasting)
        B, CTX_ = y_past.shape; total = CTX_ + H
        tgt = torch.cat([y_past, torch.zeros(B,H,device=y_past.device)], 1).unsqueeze(-1)
        obs = torch.zeros(B,total,1,dtype=torch.bool,device=y_past.device); obs[:,:CTX_,0]=True
        sid = torch.arange(B,device=y_past.device).unsqueeze(1).repeat(1,total)
        tid = torch.arange(total,device=y_past.device).unsqueeze(0).repeat(B,1)
        vid = torch.zeros(B,total,dtype=torch.long,device=y_past.device)
        pmsk= torch.zeros(B,total,dtype=torch.bool,device=y_past.device); pmsk[:,CTX_:]=True
        psz = torch.ones(B,total,dtype=torch.long,device=y_past.device)
        with autocast('cuda', enabled=(device.type=='cuda' and USE_AMP)):
            dist = model(tgt, obs, sid, tid, vid, pmsk, psz)
            m = dist.mean
            yhat = (m[:, CTX_:, :].mean(-1) if m.dim()==3 else m[:, CTX_:])[:, :H]
        ys_all.append(y_true.numpy()); yh_all.append(yhat.cpu().numpy())
    model.train()
    ys = np.concatenate(ys_all, 0); yh = np.concatenate(yh_all, 0)
    den = (np.abs(ys)+np.abs(yh))/2.0; mask = den != 0
    return float(100.0*np.mean(np.abs(yh[mask]-ys[mask]) / den[mask]))

# ---- baseline tiny-val sMAPE
smape_before = eval_smape_on(val_loader)
print(f"[canary] tiny_val sMAPE BEFORE = {smape_before:.2f}")

# ---- snapshot a few tensors to verify drift later
watch_names = []
for n,p in model.named_parameters():
    if p.requires_grad and ("self_attn.q_proj.weight" in n or "ffn.fc1.weight" in n or "in_proj.weight" in n):
        watch_names.append(n)
watch = {n: model.state_dict()[n].detach().clone() for n in watch_names}

# ---- one training loop with grad-norm prints and param drift ----
step = 0
while step < STEPS:
    for y_past, y_true in train_loader:
        y_past = y_past.to(device, non_blocking=True)
        y_true = y_true.to(device, non_blocking=True)

        tgt, obs, sid, tid, vid, pmsk, psz = build_inputs(y_past, y_true, H)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=(device.type=='cuda' and USE_AMP)):
            dist = model(tgt, obs, sid, tid, vid, pmsk, psz)
            m = dist.mean
            yhat = (m[:, y_past.shape[1]:, :].mean(-1) if m.dim()==3 else m[:, y_past.shape[1]:])[:, :H]
            loss = loss_mae(yhat, y_true)

        loss.backward()
        # print a couple grad norms early on
        if step in (0, 1, 10, 50):
            printed = 0
            for n,p in model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    g = p.grad.detach().float().norm().item()
                    if g > 0 and printed < 6:
                        print(f"[grad] {n[:45]:45s} ||g||={g:.3e}")
                        printed += 1

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        step += 1
        if step % 50 == 0 or step == 1:
            # drift check
            for wn in watch_names:
                if wn in model.state_dict():
                    d = (model.state_dict()[wn] - watch[wn]).float().norm().item()
                    print(f"[drift] {wn[:45]:45s} ||Δ||={d:.3e}")
            # tiny-val sMAPE progress
            sm = eval_smape_on(val_loader)
            print(f"[canary] step {step:04d} | train_MAE={loss.item():.4f} | tiny_val sMAPE={sm:.2f}")

        if step >= STEPS:
            break

# ---- final tiny-val sMAPE after updates
smape_after = eval_smape_on(val_loader)
print(f"[canary] tiny_val sMAPE AFTER  = {smape_after:.2f}  (Δ={smape_before - smape_after:+.2f})")


In [ ]:
# ===== Robust canary: guaranteed optimizer params, drift & sMAPE check =====
import numpy as np, torch, re
from torch import nn
from torch.utils.data import DataLoader, Subset
from uni2ts.model.moirai import MoiraiModule
from torch.amp import autocast

# ---- knobs ----
H, CTX       = 6, 96
TRAIN_N      = 2000
VAL_N        = 600
STEPS        = 250
BS           = 128
LR_BACK      = 3e-4
LR_HEAD      = 1e-3   # we'll fold "heads" into no_decay group later; LR_HEAD used if we detect heads by name
WEIGHT_DECAY = 0.01
USE_AMP      = False
SEED         = 777

np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- fixed tiny splits from existing train_ds ----
all_idx = np.arange(len(train_ds)); np.random.shuffle(all_idx)
tiny_train_idx = all_idx[:TRAIN_N]
tiny_val_idx   = all_idx[TRAIN_N:TRAIN_N+VAL_N]
tiny_train = Subset(train_ds, tiny_train_idx)
tiny_val   = Subset(train_ds, tiny_val_idx)

train_loader = DataLoader(tiny_train, batch_size=BS, shuffle=True, drop_last=True,  num_workers=0, pin_memory=(device.type=="cuda"))
val_loader   = DataLoader(tiny_val,   batch_size=256, shuffle=False, drop_last=False, num_workers=0, pin_memory=(device.type=="cuda"))

# ---- model: load fresh, unfreeze ALL (to rule out masking) ----
model = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).train()
for p in model.parameters(): p.requires_grad = True

# ---- build optimizer groups by tensor properties (no regex) ----
decay, no_decay = [], []
maybe_heads = []  # optional: if names contain typical head patterns we can boost LR
for n, p in model.named_parameters():
    if not p.requires_grad: continue
    is_bias_or_norm = (p.ndim == 1) or n.endswith(".bias") or ("norm" in n.lower())
    if is_bias_or_norm:
        no_decay.append(p)
    else:
        decay.append(p)
    if re.search(r"(in_proj|out_proj|proj|projection|final|lm_head|head)", n):
        maybe_heads.append(p)

# ensure we actually have params
trainable_cnt = sum(p.numel() for p in model.parameters() if p.requires_grad)
opt_param_cnt = sum(p.numel() for g in [decay, no_decay] for p in g)
print(f"✅ trainable params: {trainable_cnt:,} | in optimizer: {opt_param_cnt:,}")
if opt_param_cnt == 0:
    # fallback: throw everything in one group
    decay = [p for p in model.parameters() if p.requires_grad]
    no_decay = []

# build optimizer: optional head LR boost if we detected any
if len(maybe_heads) > 0:
    # separate heads into their own higher-LR, no-decay group
    no_decay = [p for p in no_decay if p not in maybe_heads]
    optimizer = torch.optim.AdamW([
        {"params": maybe_heads, "lr": LR_HEAD, "weight_decay": 0.0},
        {"params": decay,      "lr": LR_BACK, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay,   "lr": LR_BACK, "weight_decay": 0.0},
    ])
else:
    optimizer = torch.optim.AdamW([
        {"params": decay,    "lr": LR_BACK, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay, "lr": LR_BACK, "weight_decay": 0.0},
    ])

loss_mae = nn.L1Loss()
scaler   = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda" and USE_AMP))

def build_inputs_forecast(y_past, horizon):
    B, CTX_ = y_past.shape; total = CTX_ + horizon
    tgt_in = torch.cat([y_past, torch.zeros(B,horizon,device=y_past.device)], 1).unsqueeze(-1)  # inference-style
    obs = torch.zeros(B,total,1,dtype=torch.bool,device=y_past.device); obs[:,:CTX_,0]=True
    sid = torch.arange(B,device=y_past.device).unsqueeze(1).repeat(1,total)
    tid = torch.arange(total,device=y_past.device).unsqueeze(0).repeat(B,1)
    vid = torch.zeros(B,total,dtype=torch.long,device=y_past.device)
    pmsk= torch.zeros(B,total,dtype=torch.bool,device=y_past.device); pmsk[:,CTX_:]=True
    psz = torch.ones(B,total,dtype=torch.long,device=y_past.device)
    return tgt_in, obs, sid, tid, vid, pmsk, psz

@torch.no_grad()
def eval_smape_on(loader):
    model.eval()
    ys_all, yh_all = [], []
    for y_past, y_true in loader:
        y_past = y_past.to(device, non_blocking=True)
        tgt_in, obs, sid, tid, vid, pmsk, psz = build_inputs_forecast(y_past, H)
        with autocast('cuda', enabled=(device.type=='cuda' and USE_AMP)):
            dist = model(tgt_in, obs, sid, tid, vid, pmsk, psz)
            m = dist.mean
            yhat = (m[:, y_past.shape[1]:, :].mean(-1) if m.dim()==3 else m[:, y_past.shape[1]:])[:, :H]
        ys_all.append(y_true.numpy()); yh_all.append(yhat.cpu().numpy())
    model.train()
    ys = np.concatenate(ys_all, 0); yh = np.concatenate(yh_all, 0)
    den = (np.abs(ys)+np.abs(yh))/2.0; mask = den != 0
    return float(100.0*np.mean(np.abs(yh[mask]-ys[mask]) / den[mask]))

print(f"[canary*] tiny_val sMAPE BEFORE = {eval_smape_on(val_loader):.2f}")

# snapshot a few easy-to-find params
watch_names = []
for n,p in model.named_parameters():
    if p.requires_grad and (n.endswith("in_proj.weight") or "self_attn.q_proj.weight" in n or "ffn.fc1.weight" in n):
        watch_names.append(n)
        if len(watch_names) >= 6: break
watch = {n: model.state_dict()[n].detach().clone() for n in watch_names}

# training: MAE on mean (simple, stable)
step = 0
while step < STEPS:
    for y_past, y_true in train_loader:
        y_past = y_past.to(device, non_blocking=True)
        y_true = y_true.to(device, non_blocking=True)

        tgt_in, obs, sid, tid, vid, pmsk, psz = build_inputs_forecast(y_past, H)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=(device.type=='cuda' and USE_AMP)):
            dist = model(tgt_in, obs, sid, tid, vid, pmsk, psz)
            m = dist.mean
            yhat = (m[:, y_past.shape[1]:, :].mean(-1) if m.dim()==3 else m[:, y_past.shape[1]:])[:, :H]
            loss = loss_mae(yhat, y_true)

        loss.backward()

        # sanity: do we have gradients and params in optimizer?
        if step in (0, 1):
            total_with_grad = sum(1 for p in model.parameters() if p.grad is not None)
            nonzero_grads = sum(1 for p in model.parameters() if (p.grad is not None and p.grad.abs().sum().item() > 0))
            print(f"[grad] tensors with grad: {total_with_grad} | nonzero grad: {nonzero_grads}")

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # drift / metric every 50 steps
        if step % 50 == 0 or step == 1:
            for wn in watch_names:
                d = (model.state_dict()[wn] - watch[wn]).float().norm().item()
                print(f"[drift] {wn[:45]:45s} ||Δ||={d:.3e}")
            sm = eval_smape_on(val_loader)
            print(f"[canary*] step {step:04d} | train_MAE={loss.item():.4f} | tiny_val sMAPE={sm:.2f}")

        step += 1
        if step >= STEPS:
            break

print(f"[canary*] tiny_val sMAPE AFTER  = {eval_smape_on(val_loader):.2f}")
